In [ ]:
import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

In [2]:
mlflow.set_tracking_uri("http://localhost:5012")

client = MlflowClient()

In [43]:
runs = []

for run in client.search_runs(experiment_ids=[client.get_experiment_by_name("Distilbert Fine Tuning").experiment_id]):
    run_metrics = run.data.metrics
    run_info = run.info
    if 'eval_accuracy' in run_metrics and run_metrics['epoch'] >= 5:
        runs.append({
            'model_name': "Distilbert Fine Tuning",
            'train_runtime': f"{round(run_metrics['train_runtime'] / 60, 2)} min",
            'eval_accuracy': run_metrics['eval_accuracy'],
            'eval_precision': run_metrics['eval_precision'],
            'eval_recall': run_metrics['eval_recall'],
            'eval_f1': run_metrics['eval_f1'],
        })

for run in client.search_runs(experiment_ids=[client.get_experiment_by_name("TF_IDF Model [RAW]").experiment_id]):
    run_metrics = run.data.metrics
    run_info = run.info
    if 'testing_accuracy_score' in run_metrics:
        runs.append({
            'model_name': "TF_IDF Model [RAW]",
            'train_runtime': f"{round(((run.info.end_time - run.info.start_time) / 1000 / 60), 2)} min",
            'eval_accuracy': run_metrics['testing_accuracy_score'],
            'eval_recall': run_metrics['testing_recall_score'],
            'eval_precision': run_metrics['testing_precision_score'],
            'eval_f1': (2 * run_metrics['testing_recall_score'] * run_metrics['testing_precision_score']) / (run_metrics['testing_recall_score'] + run_metrics['testing_precision_score']),
        })

for run in client.search_runs(experiment_ids=[client.get_experiment_by_name("TF_IDF Model [PROCESSED]").experiment_id]):
    run_metrics = run.data.metrics
    run_info = run.info
    if 'testing_accuracy_score' in run_metrics:
        runs.append({
            'model_name': "TF_IDF Model [PROCESSED]",
            'train_runtime': f"{round(((run.info.end_time - run.info.start_time) / 1000 / 60), 2)} min",
            'eval_accuracy': run_metrics['testing_accuracy_score'],
            'eval_recall': run_metrics['testing_recall_score'],
            'eval_precision': run_metrics['testing_precision_score'],
            'eval_f1': (2 * run_metrics['testing_recall_score'] * run_metrics['testing_precision_score']) / (run_metrics['testing_recall_score'] + run_metrics['testing_precision_score']),
        })

In [47]:
df = pd.DataFrame(runs)

df.rename(columns={
    "model_name": "Model",
    "train_runtime": "Training time",
    "eval_accuracy": "Accuracy",
    "eval_f1": "Macro F1",
    "eval_precision": "Precision",
    "eval_recall": "Recall"
}, inplace=True)

df.to_csv("results/results.csv", index=False)

| Model                    | Training time   |   Accuracy |   Precision |   Recall |   Macro F1 |
|:-------------------------|:----------------|-----------:|------------:|---------:|-----------:|
| Distilbert Fine Tuning   | 24.77 min       |    0.93188 |    0.930331 |  0.93368 |   0.932002 |
| Distilbert Fine Tuning   | 24.69 min       |    0.93208 |    0.931873 |  0.93232 |   0.932096 |
| Distilbert Fine Tuning   | 24.77 min       |    0.93252 |    0.929394 |  0.93616 |   0.932765 |
| TF_IDF Model [RAW]       | 1.48 min        |    0.88292 |    0.884057 |  0.88144 |   0.882746 |
| TF_IDF Model [RAW]       | 1.47 min        |    0.88292 |    0.884057 |  0.88144 |   0.882746 |
| TF_IDF Model [RAW]       | 1.63 min        |    0.88292 |    0.884057 |  0.88144 |   0.882746 |
| TF_IDF Model [RAW]       | 5.53 min        |    0.88292 |    0.884057 |  0.88144 |   0.882746 |
| TF_IDF Model [PROCESSED] | 1.01 min        |    0.87948 |    0.877277 |  0.8824  |   0.879831 |
| TF_IDF Model [PROCESSED] | 1.01 min        |    0.87948 |    0.877277 |  0.8824  |   0.879831 |
| TF_IDF Model [PROCESSED] | 1.09 min        |    0.87948 |    0.877277 |  0.8824  |   0.879831 |